# Chapter 13: Design a Search Autocomplete System
*From "System Design Interview"*

## TL;DR

Design an autocomplete (typeahead) system that returns the **top-k most searched queries** for a given prefix in under 100 ms. The core data structure is a **trie** with top-k results cached at every node. An **offline data-gathering pipeline** aggregates query logs and rebuilds the trie periodically, while a **query service** reads from a distributed trie cache for lightning-fast lookups.

## Requirements

### Functional
- Match prefixes at the **beginning** of search queries only
- Return **top 5** suggestions ranked by historical frequency
- Support lowercase English alphabetic characters

### Non-Functional
- **< 100 ms** response time (avoid UI stuttering)
- Highly available and scalable (10M DAU)
- Suggestions updated periodically (not necessarily real-time)

## Back-of-the-Envelope Estimation

In [ ]:
# Autocomplete system estimation
dau = 10_000_000          # daily active users
searches_per_day = 10     # avg searches per user per day
chars_per_query = 20      # avg characters per query (4 words x 5 chars)
bytes_per_char = 1        # ASCII encoding

# Each keystroke triggers an autocomplete request
requests_per_search = chars_per_query  # 20 requests per search

total_daily_queries = dau * searches_per_day * requests_per_search
qps = total_daily_queries / (24 * 3600)
peak_qps = qps * 2

print(f"Daily autocomplete requests: {total_daily_queries:,.0f}")
print(f"Average QPS:                 {qps:,.0f}")
print(f"Peak QPS:                    {peak_qps:,.0f}")

# New data storage per day (20% of queries are new)
new_query_fraction = 0.20
new_data_per_day_gb = (dau * searches_per_day * bytes_per_char
                       * chars_per_query * new_query_fraction) / 1e9
print(f"\nNew data added daily:        {new_data_per_day_gb:.1f} GB")

## High-Level Design

```mermaid
graph LR
    subgraph Read Path
        U[User types prefix] --> LB[Load Balancer]
        LB --> API[API Servers]
        API --> TC[Trie Cache]
    end

    subgraph Write Path - Offline
        AL[Analytics Logs] --> AGG[Aggregators]
        AGG --> AD[Aggregated Data]
        AD --> W[Workers]
        W --> TDB[Trie DB]
        TDB --> TC
    end
```

Two decoupled paths:
| Path | Purpose | Latency |
|------|---------|---------|
| **Query Service** (read) | Return top-k for a prefix | < 100 ms |
| **Data Gathering** (write) | Aggregate logs, rebuild trie | Weekly batch |

## Deep Dive: Trie Data Structure

### Basic Trie
- Each node stores a **character** and up to **26 children**
- Root = empty string; every root-to-leaf path = a stored string
- Augment each node with **frequency** to rank results

### Naive Top-K Algorithm
1. Find prefix node: **O(p)**
2. Traverse entire subtree to collect valid children: **O(c)**
3. Sort children, return top-k: **O(c log c)**

Total: **O(p + c + c log c)** -- too slow for large tries.

### Optimized Trie (two key improvements)

| Optimization | Effect |
|---|---|
| **Limit prefix length** (e.g., 50 chars) | Find-prefix becomes O(1) |
| **Cache top-k at every node** | Return top-k in O(1), no subtree traversal |

After both optimizations, total lookup time: **O(1)**.

Trade-off: significantly more space per node, but fast response time justifies it.

## Deep Dive: Data Gathering Service

```mermaid
graph LR
    A[Analytics Logs] --> B[Aggregators]
    B --> C[Aggregated Data weekly]
    C --> D[Workers]
    D --> E[Trie DB]
    E --> F[Trie Cache distributed]
    F --> G[API Servers]
```

| Component | Role |
|-----------|------|
| **Analytics Logs** | Append-only raw query logs |
| **Aggregators** | Summarize query frequencies (weekly or real-time) |
| **Workers** | Build new trie from aggregated data |
| **Trie DB** | Persistent storage (document store or key-value store) |
| **Trie Cache** | In-memory distributed cache; weekly snapshot from DB |

## Deep Dive: Query Service Optimizations

| Optimization | Benefit |
|---|---|
| **AJAX requests** | No full page refresh; async autocomplete |
| **Browser caching** | Cache suggestions client-side (e.g., max-age=3600) |
| **Data sampling** | Log only 1-in-N queries to reduce processing load |

## Deep Dive: Trie Sharding

**Naive approach**: Shard by first character (a-m on server 1, n-z on server 2). Problem: highly uneven distribution ("c" has far more words than "x").

**Smart approach**: A **shard map manager** analyzes historical query distribution and assigns prefix ranges to shards for balanced load. E.g., "s" alone may equal "u" through "z" combined.

```mermaid
graph TD
    SM[Shard Map Manager] --> S1[Shard 1: a-e]
    SM --> S2[Shard 2: f-k]
    SM --> S3[Shard 3: l-q]
    SM --> S4[Shard 4: r-s]
    SM --> S5[Shard 5: t-z]
```

## Trade-offs

| Decision | Pro | Con |
|----------|-----|-----|
| Cache top-k at every trie node | O(1) lookup | High memory usage |
| Weekly trie rebuild vs real-time | Simple, efficient | Stale suggestions for trending queries |
| Trie in distributed cache | Ultra-fast reads | Cache miss requires DB fallback |
| Data sampling (1-in-N logging) | Reduced processing cost | Less precise frequency counts |
| Smart shard map manager | Balanced load | Added operational complexity |

## Key Takeaways

1. **Trie + top-k caching** is the go-to data structure for prefix-based autocomplete
2. **Separate read and write paths** -- query service reads cache, data pipeline rebuilds offline
3. **Browser caching and AJAX** are cheap client-side wins for reducing load
4. **Shard by query distribution**, not naively by alphabet, to avoid hotspots
5. For **real-time trending** queries, consider streaming systems (Kafka, Spark Streaming) and weighted recency in the ranking model

## See Also
- [[trie-data-structure]] | [[top-k-caching-in-trie]] | [[data-gathering-service]]
- [[query-service]] | [[trie-sharding]]